In [ ]:
import pandas as pd

# Column names based on wdbc.names (ID, diagnosis, then 30 features).
columns = [
    "id",
    "diagnosis",
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave_points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",
    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave_points_se",
    "symmetry_se",
    "fractal_dimension_se",
    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave_points_worst",
    "symmetry_worst",
    "fractal_dimension_worst",
]

df = pd.read_csv("wdbc.data", header=None, names=columns)

print("Shape:", df.shape)
print("\nDiagnosis distribution:")
print(df["diagnosis"].value_counts())

print("\nFirst 5 rows:")
print(df.head())

print("\nNumeric summary:")
print(df.describe().T)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Class balance plot
plt.figure(figsize=(6, 4))
ax = sns.countplot(
    data=df,
    x="diagnosis",
    hue="diagnosis",
    order=["B", "M"],
    palette=["#4E79A7", "#E15759"],
    legend=False,
)
ax.set_title("Diagnosis Class Balance")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Count")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

# Histograms for a few informative features
features_to_plot = ["radius_mean", "texture_mean", "perimeter_mean", "area_mean"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, feature in zip(axes.ravel(), features_to_plot):
    sns.histplot(data=df, x=feature, hue="diagnosis", kde=True, bins=30, ax=ax, element="step")
    ax.set_title(f"Distribution of {feature}")

plt.tight_layout()
plt.show()

## Build a Diagnosis Prediction Model

We train a Logistic Regression classifier to predict `diagnosis` from the 30 numeric features and evaluate it on a held-out test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Features and target
X = df.drop(columns=["id", "diagnosis"])
y = df["diagnosis"]

# Train/test split (stratify keeps class ratio in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale + Logistic Regression in one pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=42)),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Benign (B)", "Malignant (M)"]))

cm = confusion_matrix(y_test, y_pred, labels=["B", "M"])
ConfusionMatrixDisplay(cm, display_labels=["B", "M"]).plot(cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Coefficients are for the positive class (usually 'M' when classes are ['B', 'M']).
clf = model.named_steps["clf"]
coefs = clf.coef_[0]
classes = clf.classes_

importance = pd.DataFrame({
    "feature": X.columns,
    "coefficient": coefs,
})
importance["abs_coefficient"] = importance["coefficient"].abs()
importance = importance.sort_values("abs_coefficient", ascending=False)

print("Model classes order:", classes)
print("\nTop 10 most impactful features by |coefficient|:")
print(importance[["feature", "coefficient", "abs_coefficient"]].head(10).to_string(index=False))

# Visualize top features
top_n = 15
plot_df = importance.head(top_n).sort_values("coefficient")
colors = np.where(plot_df["coefficient"] >= 0, "#E15759", "#4E79A7")

plt.figure(figsize=(9, 7))
plt.barh(plot_df["feature"], plot_df["coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Top Logistic Regression Coefficients\n(+ pushes toward malignant, - toward benign)")
plt.xlabel("Coefficient value")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

# Use the same train/test split already defined in previous cells.
# RFECV finds a good feature subset with cross-validation, then we keep top 10 by ranking.
base_lr = LogisticRegression(max_iter=3000, random_state=42)
rfecv = RFECV(
    estimator=base_lr,
    step=1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
    min_features_to_select=10,
)

X_train_scaled = StandardScaler().fit_transform(X_train)
rfecv.fit(X_train_scaled, y_train)

ranking_df = pd.DataFrame({
    "feature": X.columns,
    "rank": rfecv.ranking_,
})

best_10_features = (
    ranking_df.sort_values(["rank", "feature"]).head(10)["feature"].tolist()
)

print("Selected 10 features:")
for i, f in enumerate(best_10_features, 1):
    print(f"{i}. {f}")

# Train/evaluate model with selected 10 features
small_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=3000, random_state=42)),
])

small_model.fit(X_train[best_10_features], y_train)
y_pred_10 = small_model.predict(X_test[best_10_features])

cv_scores_10 = cross_val_score(
    small_model,
    X_train[best_10_features],
    y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
)

print("\n10-feature model CV accuracy (train, 5-fold):", round(cv_scores_10.mean(), 4))
print("10-feature model test accuracy:", round(accuracy_score(y_test, y_pred_10), 4))